# Tutorial on visualization of Neural Networks

This exercise aims at exploring different ways of visualizing Neural Networks:
- t-SNE of representations (CIFAR10)
- grad-CAM (ImageNet)
- activation maximization (ImageNet)

First, some preliminaries that facilitate plotting and data access on Google drive ... just execute !

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.colors import ListedColormap

plt.rcParams['figure.figsize'] = (20.0, 20.0)
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%matplotlib inline

import importlib.util
import sys

In [ ]:
!pip install pytorch_lightning --quiet
!pip install transformers==4.46.1 --quiet


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

# Visualization of the encodings by t-SNE

First, we will visualize encodings of a network trained on the CIFAR data set. Here, we import and preprocess the data. We keep the original labels in `y_data` (to be used for visualization later on). For training, we need to transform them to one-hot-vectors.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Load CIFAR-10 dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    #transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # normalize to [-1, 1]
])

train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_data = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Separate training data into train and validation sets
train_size = 40000
val_size = len(train_data) - train_size
train_data, val_data = random_split(train_data, [train_size, val_size])

# Convert labels to one-hot encoding
def one_hot_encode(labels, num_classes=10):
    return torch.eye(num_classes)[labels]

# Load data into DataLoaders
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

We will first visualize the data.

In [ ]:
# helper function to plot a few images for each class
def plot_array(fig, dataset, classes_to_plot=None, samples_per_class=7):
    Y = dataset.targets
    if classes_to_plot is None:
        classes_to_plot = np.unique(Y)
    num_classes = len(classes_to_plot)

    for k, y in enumerate(classes_to_plot):
        idxs = np.flatnonzero(Y == y)
        idxs = np.random.choice(idxs, samples_per_class, replace=False)
        #print(y, idxs)

        for i, idx in enumerate(idxs):
            plt_idx = i * num_classes + k + 1
            ax = fig.add_subplot(samples_per_class, num_classes, plt_idx)
            image = dataset[idx][0].permute((1, 2, 0)) * 255
            ax.imshow(image.to(torch.uint8))
            ax.axis('off')
fig = plt.figure(figsize=(12, 12))
plot_array(fig, train_data.dataset, samples_per_class=10)

Next we define a neural network and train it on the data set. Here we use pytorch lightning to train the model using torch: https://lightning.ai/docs/pytorch/stable/starter/introduction.html

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl
from torchmetrics import Accuracy


class CNNModel(nn.Sequential):
    def __init__(self, num_class=10):
        super(CNNModel, self).__init__()
        # Feature extractor
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout(0.2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout(0.2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout(0.2),

            nn.AdaptiveAvgPool2d((1, 1)),  # Global average pooling
            nn.Flatten()
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_class)  # Output layer for 10 classes
        )

    def forward(self, x):
      embeddings = self.features(x)
      logits = self.classifier(embeddings)
      return logits

    def forward_features(self, x):
      return self.features(x)


# Define the Lightning Module
class LightningClassifier(pl.LightningModule):
    def __init__(self, model, learning_rate=0.001):
        super(LightningClassifier, self).__init__()
        self.model = model
        self.loss_fn = nn.CrossEntropyLoss()
        self.accuracy = Accuracy(task="multiclass", num_classes=10)
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        acc = self.accuracy(y_hat.softmax(dim=-1), y)
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        acc = self.accuracy(y_hat.softmax(dim=-1), y)
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        acc = self.accuracy(y_hat.softmax(dim=-1), y)
        self.log('test_loss', loss)
        self.log('test_acc', acc)

    def predict_step(self, batch, batch_idx):
        x, y = batch
        return self.model.forward_features(x)

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.learning_rate)

# Instantiate the Lightning model
torch.cuda.empty_cache()
model = CNNModel()
ligthning_model = LightningClassifier(model, learning_rate=0.001)
ligthning_model.to(device)

In [ ]:
import os
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

# Train the model using the PyTorch Lightning Trainer
callbacks = [
    LearningRateMonitor(logging_interval="step"),
    ModelCheckpoint(
        dirpath=os.path.join("logs", "checkpoints"),
        filename="model",
        verbose=True,
        save_top_k=1,
        save_last=False,
        save_weights_only=True)]
logger = TensorBoardLogger("logs", name="classifier")

# where to store the best model (checkpoint)
best_checkpoint_path = os.path.join("logs", "checkpoints", "model.ckpt")

# trainer instantiation
trainer = pl.Trainer(
    max_epochs=10,
    callbacks=callbacks, logger=logger,
    accelerator="gpu" if device == "cuda" else "cpu",
    devices=1 if device == "cuda" else 0, # outside colab, you can increase the number of GPUs.
    gradient_clip_val=1., gradient_clip_algorithm="norm", # gradient clipping to improve stability
    precision="16" # half precision to speed up the training
    )

In [ ]:
# This is how to train a lightning model
trainer.fit(ligthning_model, train_loader, val_loader)

In [ ]:
test_metrics = trainer.test(ligthning_model, test_loader, verbose=True, ckpt_path=best_checkpoint_path)

# to be sure that we loaded the best checkpoint
checkpoint = torch.load(best_checkpoint_path)
ligthning_model.load_state_dict(checkpoint["state_dict"], strict=True)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

Now, that we trained the neural network, we observe training and validation error, and we try to get a solution that roughly obtains 80\% or more on the test set.

Now, we need to define a few plots ... just execute !

In [ ]:
class_definition = {0: 'airplane', 1: 'automobile', 2: 'bird', 3: 'cat',
                    4: 'deer', 5: 'dog', 6: 'frog', 7: 'horse', 8: 'ship',
                    9: 'truck'}

from matplotlib.colors import to_hex

# definition of the scatterplot
def make_scatterplot(X, y, feature1=None, feature2=None,
                     class_indices=None, class_definition=None):
    if class_indices is None:
        class_indices = np.unique(y)
    if class_definition is None:
        class_definition = dict(zip(class_indices, [str(i) for i in class_indices]))
    if feature1 is None:
      feature1 = 'Component 1'
    if feature2 is None:
      feature2 = 'Component 2'

    # colors
    colors = plt.cm.get_cmap('tab10', 10).colors[:,:3]

    fig = plt.figure(figsize = (8,8))
    ax = fig.add_subplot(1,1,1)
    ax.set_xlabel(feature1)
    ax.set_ylabel(feature2)
    ax.set_title('Scatter plot: %s vs. %s' % (feature2, feature1))

    comp1 = X[:,0]
    comp2 = X[:,1]
    for class_index in class_indices:
        class_label = class_definition[class_index]
        ax.scatter(comp1[y==class_index],
                   comp2[y==class_index],
                   c=to_hex(colors[class_index]),
                   label=class_label,
                   s=15)
    ax.legend()
    ax.grid()

Now, we will extract the features of a layer and visualize the distribution of the encodings with t-SNE.

First, we start with layer `flatten`. This is the last layer before the dense layers in the network.

In [ ]:
model = ligthning_model.model.eval().to(device)


# features_upstream and features_downstream will hold the two feature
# vectors we are going to look at later.
features_upstream = []
features_downstream = []
y = []
for data in val_loader:
  x_batch, y_batch = data
  y.append(y_batch)
  with torch.no_grad():

    # try to understand what is happening here.
    features_batch = model.features(x_batch.to(device))
    features_batch_ds = model.classifier[1](model.classifier[0](features_batch))

  features_upstream.append(features_batch.cpu())
  features_downstream.append(features_batch_ds.cpu())

# output --> vector
y = torch.hstack(y).numpy()

# we consider a subsample of the entire dataset
idxs_sampled = np.random.choice(np.arange(len(y)), 2000, replace=False)

# output subsampling
y = y[idxs_sampled]

# we choose whether to use features_upstream or features_downstream
features = torch.vstack(features_upstream).numpy()
features = features[idxs_sampled]


Finally, we perform t-SNE.

In [ ]:
from sklearn.manifold import TSNE
X_embedded = TSNE(n_components=2, perplexity=30).fit_transform(features)
make_scatterplot(X_embedded, y.flatten(), class_definition=class_definition)

**Assignment**: Explain what is a feature vector (hint: other terms in the literature are "Encodings", "Embeddings")

**Assignment**: Try out several perplexities: 5, 10, 30. What do you observe?

**Assignment**: Visualize now the embeddings at the two fully connected layers (in a different cell). Do you observe differences (albeit subtle)? Imagine you would like to use the same representations in another project (same image size, but other classes). Which of the representations seems more useful? Why?

**Assignment**: Visualize the tSNE plot of the untrained embeddings (model output before training). What do you observe?

# Classification activation maps (grad-CAM)

Classification activation maps provide certainly the most popular visualization methods for network inspection.

In [ ]:
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
!pip install gdown
!gdown --id 1-ypc0JK1Hp4d3dfO6eq0mPcx669PthXw --output /content/imagenet.zip
!unzip -q /content/imagenet.zip
!rm /content/imagenet.zip

In [ ]:
!pip install torchcam

Next, we will load VGG16, pretrained on `ImageNet`. This is the network we are going to investigate.

In [ ]:
from torchvision import models
import torch.nn.functional as F

torch.cuda.empty_cache()
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
model = model.eval().to(device)

labels_names = models.VGG16_Weights.IMAGENET1K_V1.meta["categories"]

Now, we load an image from the `ImageNet` data base. We have downloaded these images: they are in the folder `imagenet`.

In [ ]:
import os
from PIL import Image

filename = 'bridge1.jpg'
folder_name = '/content/imagenet'

# animals:
# bird : filename = '418657219_3567961db1.jpg'
# dog : filename = '425248370_b15374000e.jpg'
# bird : filename = '485627874_8f4144223a.jpg'

# bridges:
# filename = 'bridge1.jpg'
# filename = 'bridge2.jpg'
# filename = 'bridge3.jpg'

image = Image.open(os.path.join(folder_name, filename)).resize((224, 224))
plt.title('Bird 1')
plt.imshow(image)


Now, we will predict the label of the image.

In [ ]:
from torchvision import transforms


preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])  # Normalize to match ImageNet
])

img_prep = preprocess(image)
img_prep = img_prep.unsqueeze(0)

with torch.no_grad():
    img_prediction = model(img_prep.to(device))

# Get probabilities by applying softmax to the output
probs = F.softmax(img_prediction, dim=1)

# Convert the probabilities to a numpy array and get the top 3 predictions
top_probs, top_indices = torch.topk(probs, 3, dim=1)
top_probs = top_probs.squeeze().cpu().numpy()
top_indices = top_indices.squeeze().cpu().numpy()

for i in range(3):
    label = labels_names[top_indices[i]]
    print(f"{label} ({top_indices[i]}): {top_probs[i] * 100:.2f}%")

# Get the solution index with the maximum probability
max_index = top_indices[0]
print('Predicted Index:', max_index)

We will use the [Grad-Cam++](https://arxiv.org/pdf/1710.11063) implementation from torch-cam https://github.com/frgfm/torch-cam

Note that the output of the model needs to be the the logits and not probabilities (default behaviour in torchvision mdeols). The reason is that with `softmax` we cannot study the influence of neurons on the output $y_k$, if the output depends on all classes (which is the case when we use `softmax`).

In [ ]:
from torchcam.methods import GradCAMpp
from torchcam.utils import overlay_mask
from torchvision.transforms.functional import to_pil_image

n=3

img_prep = preprocess(image)
img_prep = img_prep.unsqueeze(0)

for k in range(n):
  with GradCAMpp(model) as cam_extractor:
      # Preprocess your data and feed it to the model
      out = model(img_prep.to(device))
      # Retrieve the CAM by passing the class index and the model output
      indices = out.squeeze(0).topk(n).indices
      print(int(indices[k]))
      activation_map = cam_extractor(int(indices[k]), out)

      # Resize the CAM and overlay it
      result = overlay_mask(image, to_pil_image(activation_map[0].squeeze(0), mode='F'), alpha=0.5)
      # Display it
      plt.imshow(result); plt.axis('off'); plt.tight_layout(); plt.show()

**Assignment:** Test the grad-CAM first on the three animal images, and verify that you obtain a reasonable result. Then test the algorithm on the three bridge images. Visualize the top-3 predictions.(Make a nice plot with the results)

*   For `Bridge1.jpg` you obtain a wrong classification, but what can be said about the learned network in view of the visualization of the top-3 predictions?
*   For `Bridge2.jpg` you get the right result, but what can you say about the "understanding" of the image in view of your visualization result?

Note that the correspondence of class names and indices can be found at:
[ImageNet](https://gist.github.com/yrevar/942d3a0ac09ec9e5eb3a)


# Activation Maximization

So far, we visualized activations of images, i.e. we focused on visualizations of inner network representations for given image data.

We can also visualize properties of the network itself. A popular method is the activation maximization, where we seek an image that would maximize a given neuron inside the network.

For this, we solve the maximization problem:
\begin{equation}
x^{\ast} = {\arg \max}_{x} z(x)
\end{equation}
where $z(x)$ is the value of an arbitrary neuron (or a set of neurons, e.g. the neurons in one feature map) in the network.

Typically, $z(x)=S_c(x)$ is the value of the output layer for one particular class. We therefore seek the image that maximizes the output for a particular class (e.g. the output for `water_ouzel`, `index: 20`).

More information here: https://distill.pub/2017/feature-visualization/

In [ ]:
#@title Maximum activation visualization functions

# Inspired from  tf-keras-vis https://github.com/keisen/tf-keras-vis/blob/c793148ff6bc97d03880b107b958fb9c4f87efc5/tf_keras_vis/activation_maximization/__init__.py#L17

import torch.nn as nn
import numpy as np


class TotalVariation2D(nn.Module):
    """A regularizer that introduces Total Variation."""

    def __init__(self, weight=10.0, name='TotalVariation2D'):
        """
        Args:
            weight: This value will be applied to TotalVariation values.
                Defaults to 10.0.
            name : Instance name.
                Defaults to 'TotalVariation2D'.
        """
        super(TotalVariation2D, self).__init__()
        self.weight = float(weight)

    def forward(self, input_value):
        if len(input_value.shape) != 4:
            raise ValueError("Input shape must be (batch_size, channels, height, width), "
                             f"but was {input_value.shape}.")

        # Calculate total variation
        tv = torch.sum(torch.abs(input_value[:, :, 1:, :] - input_value[:, :, :-1, :])) + \
             torch.sum(torch.abs(input_value[:, :, :, 1:] - input_value[:, :, :, :-1]))

        # Normalize by the number of elements and apply weight
        tv /= float(input_value.shape[1] * input_value.shape[2] * input_value.shape[3])
        tv *= self.weight
        return tv


class Norm(nn.Module):
    """A regularizer that introduces Norm."""

    def __init__(self, weight=10.0, p=2, name='Norm'):
        """
        Args:
            weight: This weight will be applied to the norm values.
                Defaults to 10.
            p: Order of the norm. Defaults to 2.
            name: Instance name. Defaults to 'Norm'.
        """
        super(Norm, self).__init__()
        self.weight = float(weight)
        self.p = int(p)

    def forward(self, input_value):
        # Reshape to (batch_size, -1) for calculating norm
        reshaped_input = input_value.view(input_value.shape[0], -1)

        # Compute the norm along the specified dimension
        norm = torch.norm(reshaped_input, p=self.p, dim=1)

        # Normalize without in-place operations
        norm = norm / (reshaped_input.shape[1] ** (1.0 / float(self.p)))
        norm = norm * self.weight

        # Return the norm as a scalar by taking the mean
        return norm.mean()


class Jitter:
    """An input modifier that introduces random jitter.
       Jitter has been shown to produce crisper activation maximization images.
    """

    def __init__(self, jitter=8):
        """
        Args:
            jitter: The amount of jitter to apply. Defaults to 8.
        """
        self.jitter = int(jitter)

    def __call__(self, seed_input):
        ndim = len(seed_input.shape)
        if ndim < 3:
            raise ValueError("The dimensions of seed_input must be 3 or more "
                             f"(batch_size, ..., channels), but was {ndim}.")

        # Generate random shifts within the jitter range for each spatial dimension
        shifts = tuple(np.random.randint(-self.jitter, self.jitter) for _ in range(ndim - 2))
        axes = tuple(range(1, ndim - 1))  # Apply shifts to all spatial dimensions

        # Apply torch.roll to create jitter
        seed_input = torch.roll(seed_input, shifts=shifts, dims=axes)

        return seed_input


class Rotate2D:
    """An input modifier for 2D that introduces random rotation."""

    def __init__(self, degree=3.0):
        """
        Args:
            degree: The maximum degree of rotation to apply.
        """
        self.degree = float(degree)
        self.random_generator = np.random.default_rng()

    def __call__(self, seed_input):
        # Check input dimensions
        if len(seed_input.shape) != 4:
            raise ValueError("Input shape must be (batch_size, channels, height, width), "
                             f"but was {seed_input.shape}.")

        # Get a random rotation angle within the specified range
        angle = self.random_generator.uniform(-self.degree, self.degree)

        # Rotation requires transformation matrix
        theta = torch.tensor([
            [torch.cos(torch.tensor(angle * np.pi / 180)), -torch.sin(torch.tensor(angle * np.pi / 180)), 0],
            [torch.sin(torch.tensor(angle * np.pi / 180)),  torch.cos(torch.tensor(angle * np.pi / 180)), 0]
        ], dtype=seed_input.dtype, device=seed_input.device)

        # Expand to a batch of transformation matrices
        theta = theta.unsqueeze(0).repeat(seed_input.size(0), 1, 1)

        # Create affine grid for rotation and apply it with grid_sample
        grid = F.affine_grid(theta, seed_input.size(), align_corners=False)
        rotated_input = F.grid_sample(seed_input, grid, align_corners=False, padding_mode='reflection')

        return rotated_input



class ActivationMaximization:
    def __init__(self, model, lr=5e-2, plot_iterations=100):
        """
        Args:
            model: The neural network model.
            neuron_index: Index of the neuron to visualize.
            input_shape: Shape of the input image. Defaults to (224, 224).
            lr: Learning rate for optimization. Defaults to 5e-2.
            tv_loss_weight: Weight for the total variation loss. Defaults to 1e-5.
            num_iterations: Number of iterations for activation maximization. Defaults to 4000.
            plot_iterations: Number of iterations between plot updates. Defaults to 100.
        """
        self.model = model
        self.lr = lr
        self.total_var = TotalVariation2D(weight=5.)
        self.norm = Norm(weight=1.)
        self.jitter_transform = Jitter(jitter=4)
        self.rotate = Rotate2D(degree=1.)
        self.plot_iterations = plot_iterations
        self.to_pil = transforms.ToPILImage()

        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.mean_imagenet = torch.tensor([0.485, 0.456, 0.406]).view((1, 3, 1, 1)).to(device)
        self.std_imagenet = torch.tensor([0.229, 0.224, 0.225]).view((1, 3, 1, 1)).to(device)

    @staticmethod
    def total_variation_loss(x):
        """Calculates the total variation loss for an input tensor."""
        tv_loss = torch.sum(torch.abs(x[:, :, 1:, :] - x[:, :, :-1, :])) + \
                  torch.sum(torch.abs(x[:, :, :, 1:] - x[:, :, :, :-1]))
        return tv_loss

    def apply_norm(self, x):
        """Applies normalization to the input tensor (if necessary)."""
        # Placeholder for normalization; adjust as needed for your model
        return (x - self.mean_imagenet) / self.std_imagenet

    def visualize(self, neuron_index, input_shape=(224, 224), num_iterations=2000):
        tensor_shape = (1, 3) + input_shape
        input_tensor = torch.randn(tensor_shape, requires_grad=True, device="cuda")
        #optimizer = torch.optim.Adam([input_tensor], lr=self.lr)
        optimizer = torch.optim.RMSprop([input_tensor], lr=self.lr, alpha=0.999)

        # Activation Maximization Loop
        for i in range(num_iterations):
            optimizer.zero_grad()

            # Forward pass for neuron activation
            with torch.no_grad():
                input_tensor.copy_(self.jitter_transform(input_tensor))
                input_tensor.copy_(self.rotate(input_tensor))
            neuron_activation = self.model(self.apply_norm(input_tensor))[..., neuron_index]
            activation_loss = -neuron_activation.mean()  # Negative to maximize

            # Total variation loss
            total_var = self.total_var(input_tensor)
            norm = self.norm(input_tensor)
            regularizer_loss = total_var + norm
            loss = activation_loss + regularizer_loss

            # Backward and optimize
            loss.backward()
            optimizer.step()

            # Clamp values to the display range [0, 1]
            input_tensor.data = torch.clamp(input_tensor.data, 0, 1)

            # Logging or visualization step
            if i % self.plot_iterations == 0:
                print(f"Iteration {i}/{num_iterations}, Activation: {-activation_loss.item()}, TotalVar: {total_var.item()}, TotalVar: {norm.item()}")

        # Display the final maximized image
        optimized_image = input_tensor.squeeze().cpu().detach().clamp(0, 1)
        plt.imshow(self.to_pil(optimized_image))
        plt.title(f"Activation Maximization for Neuron {neuron_index}")
        plt.axis("off")
        plt.show()

In [ ]:
neuron_index = 20

# other indices to look at: 385: Indian elephant, 2: great white shark, 555: fire truck
# neuron_index = labels_names.index("Indian elephant")

activation_maximization = ActivationMaximization(model)
activation_maximization.visualize(neuron_index, num_iterations=2000, input_shape=(448, 448))

**Assignment**: You might want to play with this, e.g. 385: Indian elephant, 2: great white shark, 555: fire truck

Other classes can be found at:
[ImageNet](https://gist.github.com/yrevar/942d3a0ac09ec9e5eb3a)

**Assignment:** Generate the activation maximization image the the same index, for a network initialized randomly. What do you observe?